# Tesseract Baseline

Thin Colab/local entrypoint for OCR baseline execution.
Reusable logic stays in `src/`; this notebook only prepares runtime, uploads a sample, runs the pipeline, and exports artifacts.


In [1]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

IS_COLAB = 'google.colab' in sys.modules
REPO_URL = os.environ.get('NFSE_REPO_URL', 'https://github.com/LuizCarlosGoulart/DolpII.git')
REPO_ROOT = Path(os.environ.get('NFSE_REPO_ROOT', '/content/nfse-extractor' if IS_COLAB else Path.cwd().resolve().parent))
PROJECT_ROOT = Path(os.environ.get('NFSE_PROJECT_ROOT', str(REPO_ROOT / 'nfse-extractor' if IS_COLAB else REPO_ROOT)))

CONFIG = {
    'clone_or_pull': os.environ.get('NFSE_CLONE_OR_PULL', '1') == '1',
    'bootstrap_runtime': os.environ.get('NFSE_BOOTSTRAP_RUNTIME', '1') == '1',
    'mount_drive': os.environ.get('NFSE_MOUNT_DRIVE', '0') == '1',
    'sample_path': os.environ.get('NFSE_SAMPLE_PATH', ''),
    'language': os.environ.get('NFSE_TESSERACT_LANGUAGE', 'por'),
    'raw_output_path': os.environ.get('NFSE_TESSERACT_RAW_OUTPUT', '/content/tesseract_raw_artifacts.json' if IS_COLAB else str(PROJECT_ROOT / 'artifacts' / 'tesseract_raw_artifacts.json')),
    'candidate_output_path': os.environ.get('NFSE_TESSERACT_CANDIDATE_OUTPUT', '/content/tesseract_field_candidates.json' if IS_COLAB else str(PROJECT_ROOT / 'artifacts' / 'tesseract_field_candidates.json')),
    'preview_limit': int(os.environ.get('NFSE_PREVIEW_LIMIT', '80')),
}

print(f'IS_COLAB: {IS_COLAB}')
print(f'REPO_ROOT: {REPO_ROOT}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')


IS_COLAB: True
REPO_ROOT: /content/nfse-extractor
PROJECT_ROOT: /content/nfse-extractor/nfse-extractor


In [2]:
if IS_COLAB and CONFIG['clone_or_pull']:
    if REPO_ROOT.exists():
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull'], check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f'Project root not found: {PROJECT_ROOT}. Check NFSE_PROJECT_ROOT or the cloned repository layout.'
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if IS_COLAB and CONFIG['mount_drive']:
    from google.colab import drive
    drive.mount('/content/drive')

if IS_COLAB and CONFIG['bootstrap_runtime']:
    subprocess.run(['bash', str(PROJECT_ROOT / 'scripts' / 'colab_bootstrap.sh')], check=True)

print('Runtime ready')


Runtime ready


In [3]:
if IS_COLAB and not CONFIG['sample_path']:
    from google.colab import files

    uploaded = files.upload()
    if uploaded:
        first_file = next(iter(uploaded))
        uploaded_path = Path('/content') / first_file
        if not uploaded_path.exists():
            matches = sorted(Path.cwd().glob(first_file)) + sorted(Path("/content").rglob(first_file))
            uploaded_path = matches[-1] if matches else uploaded_path
        CONFIG['sample_path'] = str(uploaded_path)

if not CONFIG['sample_path']:
    raise ValueError('Set NFSE_SAMPLE_PATH or upload a sample PDF/image before running the pipeline.')

sample_path = Path(CONFIG['sample_path']).expanduser()
if not sample_path.exists():
    raise FileNotFoundError(f'Sample file not found: {sample_path}')

print(f'Sample: {sample_path}')


Saving 01.PDF to 01.PDF
Sample: /content/01.PDF


In [4]:
from src.engines import TesseractExtractionAdapter
from src.export import write_extracted_elements_json
from src.ingestion import load_document
from src.preprocessing import preprocess_document

normalization_module = importlib.import_module("src.normalization")
ConfigDrivenOutputNormalizer = getattr(normalization_module, "ConfigDrivenOutputNormalizer", None)

document = load_document(sample_path)
preprocessed = preprocess_document(document)
adapter = TesseractExtractionAdapter(language=CONFIG['language'])
artifacts = adapter.extract_preprocessed(preprocessed)
raw_output_path = write_extracted_elements_json(artifacts, CONFIG['raw_output_path'])

candidates = []
candidate_output_path = None
if ConfigDrivenOutputNormalizer is not None:
    normalizer = ConfigDrivenOutputNormalizer()
    candidates = normalizer.normalize(document, artifacts)
    candidate_payload = [candidate.model_dump(mode="json") for candidate in candidates]
    candidate_output_path = Path(CONFIG['candidate_output_path'])
    candidate_output_path.parent.mkdir(parents=True, exist_ok=True)
    candidate_output_path.write_text(json.dumps(candidate_payload, indent=2, ensure_ascii=False), encoding="utf-8")

print(f'Document: {document.document_id}')
print(f'Media type: {document.media_type}')
print(f'Pages: {preprocessed.metadata["page_count"]}')
print(f'Raw artifacts: {len(artifacts)}')
print(f'Field candidates: {len(candidates)}')
print(f'Raw output: {raw_output_path}')
print(f'Candidate output: {candidate_output_path}')


Document: 990a3b02-e774-5a4c-95f2-677b7a475682
Media type: application/pdf
Pages: 1
Raw artifacts: 376
Field candidates: 33
Raw output: /content/tesseract_raw_artifacts.json
Candidate output: /content/tesseract_field_candidates.json


In [5]:
print('Raw OCR preview')
for item in artifacts[: CONFIG['preview_limit']]:
    print(
        f'{item.text!r} | conf={item.confidence} | page={item.page_number} | bbox={item.bounding_box}'
    )

if candidates:
    print('\nField candidate preview')
    for candidate in candidates[: CONFIG['preview_limit']]:
        print(
            f'{candidate.field_name} = {candidate.value!r} | '
            f'conf={candidate.confidence} | '
            f'section={candidate.metadata.get("section_name")} | '
            f'label={candidate.metadata.get("label_text")}'
        )


Raw OCR preview
'Esta' | conf=0.96 | page=1 | bbox=(61.0, 57.0, 43.0, 18.0)
'nota' | conf=0.93 | page=1 | bbox=(109.0, 59.0, 42.0, 16.0)
'fiscal' | conf=0.93 | page=1 | bbox=(156.0, 57.0, 50.0, 18.0)
'foi' | conf=0.96 | page=1 | bbox=(212.0, 57.0, 21.0, 18.0)
'assinada' | conf=0.96 | page=1 | bbox=(240.0, 57.0, 89.0, 18.0)
'digitalmente' | conf=0.96 | page=1 | bbox=(334.0, 57.0, 119.0, 22.0)
'utilizando' | conf=0.96 | page=1 | bbox=(460.0, 57.0, 92.0, 18.0)
'um' | conf=0.96 | page=1 | bbox=(559.0, 62.0, 28.0, 13.0)
'certificado' | conf=0.96 | page=1 | bbox=(593.0, 57.0, 101.0, 18.0)
'ICP-Brasil.' | conf=0.94 | page=1 | bbox=(701.0, 57.0, 103.0, 18.0)
'º' | conf=0.0 | page=1 | bbox=(773.0, 121.0, 10.0, 2.0)
'.' | conf=0.35 | page=1 | bbox=(893.0, 121.0, 4.0, 3.0)
'Número' | conf=0.92 | page=1 | bbox=(1174.0, 90.0, 77.0, 31.0)
'do' | conf=0.86 | page=1 | bbox=(1256.0, 94.0, 23.0, 18.0)
'RPS' | conf=0.86 | page=1 | bbox=(1285.0, 94.0, 43.0, 18.0)
'Número' | conf=0.96 | page=1 | bbox=(1394

In [6]:
from pathlib import Path
import src.normalization as normalization

print("normalization file:", normalization.__file__)
print("has ConfigDrivenOutputNormalizer:", hasattr(normalization, "ConfigDrivenOutputNormalizer"))
print("candidates exists:", "candidates" in globals())
print("candidates len:", len(candidates) if "candidates" in globals() else None)
print("output_normalizer.py exists:", Path("/content/nfse-extractor/nfse-extractor/src/normalization/output_normalizer.py").exists())


normalization file: /content/nfse-extractor/nfse-extractor/src/normalization/__init__.py
has ConfigDrivenOutputNormalizer: True
candidates exists: True
candidates len: 33
output_normalizer.py exists: True


In [8]:
important_fields = {
    "nfse_number",
    "nfse_series",
    "issue_date",
    "verification_code",
    "provider_name",
    "provider_document",
    "provider_municipal_registration",
    "provider_address",
    "provider_email",
    "provider_uf",
    "provider_phone",
    "recipient_name",
    "recipient_document",
    "recipient_municipal_registration",
    "recipient_address",
    "recipient_email",
    "recipient_uf",
    "recipient_phone",
    "service_description",
    "service_code",
    "operation_nature",
    "service_city",
    "gross_amount",
    "taxable_amount",
    "iss_rate",
    "iss_amount",
    "iss_withheld_amount",
    "net_amount",
}

for candidate in candidates:
    if candidate.field_name in important_fields:
        print(
            f"{candidate.field_name} = {candidate.value!r} | "
            f"conf={candidate.confidence} | "
            f"section={candidate.metadata.get('section_name')} | "
            f"label={candidate.metadata.get('label_text')} | "
            f"line={candidate.metadata.get('line_text')}"
        )


gross_amount = 'R$ 160,87' | conf=0.7949999999999999 | section=values | label=valor_bruto | line=Valor bruto = R$ 160,87 Valor Liquido = R$ 156,04
iss_amount = '0,00 0,00 0,00 160,87 4,83' | conf=1.0 | section=values | label=valor_iss | line=Desc. condicionado(R$) Desc. incondicionado(R$) Deduções(R$) Base de cálculo(R$) Valor ISS(R$)
issue_date = 'da nota' | conf=1.0 | section=values | label=emissao | line=Data da emissão da nota
net_amount = 'R$ 156,04' | conf=1.0 | section=values | label=valor liquido | line=Valor bruto = R$ 160,87 Valor Liquido = R$ 156,04
nfse_number = 'desta desta foi NFS-e' | conf=1.0 | section=recipient | label=nfs-e | line=O Situação Esta Inscr.Estadual ISS NFS-e desta desta foi NFS-e Tomador emitida NFS-e: é devido com Retida ISENTO respaldo fora deste na Município Lei Nro. 49 de 14 de novembro de 2011 e regulamentada pelo Decreto de Nro. 3482/2014
nfse_number = 'Data do fato gerador' | conf=1.0 | section=service | label=nfs-e | line=Nota Fiscal Eletrônica de

In [9]:
from collections import Counter

counts = Counter(candidate.field_name for candidate in candidates)

for field_name, count in sorted(counts.items()):
    print(field_name, count)


deductions_amount 1
gross_amount 1
iss_amount 1
issue_date 1
net_amount 1
nfse_number 2
operation_nature 1
other_retentions_amount 1
provider_address 1
provider_document 2
provider_email 1
provider_municipal_registration 1
provider_name 2
provider_phone 1
provider_uf 1
recipient_address 1
recipient_document 1
recipient_email 1
recipient_municipal_registration 1
recipient_name 3
recipient_phone 1
recipient_uf 1
service_description 2
taxable_amount 2
verification_code 2
